In [2]:
! apt-get install openjdk-8-jdk-headless -qq > /dev/null
# ! wget -q https://archive.apache.org/dist/spark/spark-3.5.4/spark-3.5.4-bin-hadoop3.tgz
! cp drive/MyDrive/MMDS-data/spark-3.5.4-bin-hadoop3.tgz .
! tar xf spark-3.5.4-bin-hadoop3.tgz
! pip install -q findspark

In [3]:
! du -sh spark-3.5.4-bin-hadoop3.tgz

383M	spark-3.5.4-bin-hadoop3.tgz


In [4]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.4-bin-hadoop3"

In [5]:
import pyspark as spark
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
print(spark.__version__)

3.5.1


In [6]:
spark = SparkSession.builder \
    .appName("MMDS_2c") \
    .getOrCreate()

In [7]:
file_path = "/content/drive/MyDrive/MMDS-data/baskets.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)
basket=df.groupBy("Member_number", "date").agg(f.collect_list("itemDescription").alias("items")).orderBy("Date")

In [11]:
basket.show(truncate=False)
print(basket.count())

+-------------+----------+-------------------------------------------------+
|Member_number|date      |items                                            |
+-------------+----------+-------------------------------------------------+
|2610         |01/01/2014|[hamburger meat, bottled beer, domestic eggs]    |
|4942         |01/01/2014|[butter, frozen vegetables]                      |
|1440         |01/01/2014|[other vegetables, yogurt]                       |
|1659         |01/01/2014|[specialty chocolate, frozen vegetables]         |
|1789         |01/01/2014|[hamburger meat, candles]                        |
|2542         |01/01/2014|[sliced cheese, bottled water]                   |
|1249         |01/01/2014|[citrus fruit, coffee]                           |
|2226         |01/01/2014|[sausage, bottled water]                         |
|2237         |01/01/2014|[bottled water, Instant food products]           |
|2351         |01/01/2014|[cleaner, shopping bags]                         |

In [9]:
class PCY:
  def __init__(self, basket, num_buckets, s, c):
    self.num_buckets = num_buckets
    self.df = basket
    self.s = s
    self.c = c
  def hash_pair(self, item1, item2):
    return (
        (f.abs(
            f.hash(
                f.concat(f.least(item1, item2), f.lit(","), f.greatest(item1, item2))
            ) * 37
        ) % 1000000007) % self.num_buckets
    )
  def pair_count_df(self):
    pair_counts_df = self.df.select(
        f.posexplode("items").alias("idx1", "item1"),
        f.posexplode("items").alias("idx2", "item2")
    ).filter(
        (f.col("idx1") < f.col("idx2")) &
        (f.col("item1") != f.col("item2"))
    ).select(
        f.concat(
            f.least("item1", "item2"),
            f.lit(","),
            f.greatest("item1", "item2")
        ).alias("pair")
    ).groupBy("pair").agg(
        f.count("*").alias("pair_count")
    )
    return pair_counts_df
  def fre_pair(self):
    f_i = self.fre_item().select("item")
    bucket = self.pass1()
    pair_count = self.pair_count_df()
    fre_b = bucket.filter(f.col("bucket_count") >= self.s)
    split = pair_count.select(
        f.split(f.col("pair"), ",").alias("items"),"pair_count"
    ).select(
        f.col("items")[0].alias("a"),
        f.col("items")[1].alias("b"),
        f.col("pair_count")
    )
    hash_pair = split.select(
        f.col("a"),
        f.col("b"),
        f.col("pair_count")
        ).withColumn(
        "hash_bucket", self.hash_pair(f.col("a"), f.col("b"))
    )
    freq_pair = (
        hash_pair.join(f_i.alias("items_a"), f.col("a") == f.col("items_a.item"), "inner")
             .join(f_i.alias("items_b"), f.col("b") == f.col("items_b.item"), "inner")
             .join(fre_b, f.col("bucket") == f.col("hash_bucket"), "inner")
             .filter(
                 (f.col("pair_count") >= self.s)
             )
             .select(f.col("a"), f.col("b"), f.col("pair_count"))
    )
    fre_pair = freq_pair.select(f.concat(f.col("a"), f.lit(","), f.col("b")).alias("pair"))
    return fre_pair
  def pass1(self):
    bucket_counts_df = basket.select(
            f.posexplode("items").alias("idx1", "item1"),
            f.posexplode("items").alias("idx2", "item2")
        ).filter((f.col("idx1") < f.col("idx2")) & (f.col("item1") != f.col("item2"))).select(
            f.least("item1", "item2").alias("item1"),
            f.greatest("item1", "item2").alias("item2"),
        ).distinct().withColumn(
        "bucket", self.hash_pair(f.col("item1"), f.col("item2"))
        ).groupBy("bucket").agg(
            f.count("*").alias("bucket_count"))
    self.bucket = bucket_counts_df
    return self.bucket
  def fre_item (self):
    item_count = basket.select(f.explode("items").alias("item")).groupBy("item").agg(f.count("*").alias("count"))
    self.fre_item = item_count.filter(f.col("count") >= self.s)
    return self.fre_item
  def a_rule(self):
    pair_counts = self.pair_count_df()
    self.rules = []
    item_support_df = basket.select(
        f.explode("items").alias("item")
    ).groupBy("item").agg(
        f.count("*").alias("support")
    )
    exploded_pairs_df = pair_counts.select(
        f.split(f.col("pair"), ",").alias("items"),"pair_count"
    ).select(
        f.col("items")[0].alias("a"),
        f.col("items")[1].alias("b"),
        f.col("pair_count")
    )
    rules_df = exploded_pairs_df.join(
        item_support_df.alias("supp_a"),
        f.col("a") == f.col("supp_a.item"), "left"
    ).join(
        item_support_df.alias("supp_b"),
        f.col("b") == f.col("supp_b.item"), "left"
    ).select(
        f.col("a"),
        f.col("b"),
        f.col("pair_count"),
        f.col("supp_a.support").alias("sup_a"),
        f.col("supp_b.support").alias("sup_b")
    )
    conf_df = rules_df.select(
        f.col("a"),
        f.col("b"),
        f.col("pair_count"),
        (f.col("pair_count") / f.col("sup_a")).alias("conf_ab"),
        (f.col("pair_count") / f.col("sup_b")).alias("conf_ba")
    ).filter(
        f.col("pair_count") >= self.s
    )
    for row in conf_df.collect():
        a, b = row.a, row.b
        conf_ab, conf_ba = row.conf_ab, row.conf_ba
        if conf_ab >= self.c:
            self.rules.append(f'{a} => {b}, {conf_ab:.2f}')
        if conf_ba >= self.c:
            self.rules.append(f'{b} => {a}, {conf_ba:.2f}')

    return self.rules

In [10]:
bucket_size = 100
support = 10
confidence = 0.05
item_count = []
pcy = PCY(basket, bucket_size, support, confidence)
frequent_pair = pcy.fre_pair()
association_rule = pcy.a_rule()
frequent_pair.show(truncate=False)
for rule in association_rule:
  print(rule)


+------------------------------+
|pair                          |
+------------------------------+
|coffee,pastry                 |
|beef,margarine                |
|curd,frozen meals             |
|pot plants,whole milk         |
|brown bread,curd              |
|tropical fruit,whole milk     |
|domestic eggs,shopping bags   |
|frozen vegetables,pork        |
|brown bread,pip fruit         |
|whipped/sour cream,white bread|
|cat food,fruit/vegetable juice|
|canned beer,whole milk        |
|shopping bags,soda            |
|beef,pip fruit                |
|pastry,rolls/buns             |
|detergent,rolls/buns          |
|dessert,newspapers            |
|rolls/buns,salty snack        |
|butter milk,canned beer       |
|chocolate,root vegetables     |
+------------------------------+
only showing top 20 rows

pot plants => whole milk, 0.15
tropical fruit => whole milk, 0.13
whole milk => tropical fruit, 0.05
cat food => fruit/vegetable juice, 0.06
canned beer => whole milk, 0.14
shopping 